# 10 Machine Learning — Reference Solutions

Complete solutions for the machine learning exercises on the Legionnaires' disease cluster at Songbai Nursing Home.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.inspection import permutation_importance

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = ((df["hospitalized"] == 1) | (df["outcome"] == "dead")).astype(int)

# Feature definitions
num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]
feature_cols = num_cols + cat_cols + bin_cols

X = df[feature_cols]
y_infected = df["infected"]
y_severe = df["severe_outcome"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

## Question 1: The effect of class_weight="balanced"

In [ ]:
# Without class_weight
clf_default = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=500, random_state=42)),
])
scores_default = cross_val_score(clf_default, X, y_severe, cv=5, scoring="roc_auc")

# With class_weight="balanced"
clf_balanced = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=500, random_state=42,
                                 class_weight="balanced")),
])
scores_balanced = cross_val_score(clf_balanced, X, y_severe, cv=5, scoring="roc_auc")

print("=== Task B (severe_outcome) ===")
print(f"class_weight=None:       AUC = {scores_default.mean():.3f} ± {scores_default.std():.3f}")
print(f"class_weight='balanced': AUC = {scores_balanced.mean():.3f} ± {scores_balanced.std():.3f}")

print("\n→ class_weight='balanced' gives higher weight to the minority class")
print("→ The difference in AUC is usually small, but it helps recall")
print("→ When the positive-class share is very low (e.g. <10%), the effect of balanced is more pronounced")

## Question 2: Feature importance for Task B

In [ ]:
# Random Forest on Task B
clf_rf = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])

scores_rf_b = cross_val_score(clf_rf, X, y_severe, cv=5, scoring="roc_auc")
print(f"Random Forest 5-fold CV AUC (Task B) = {scores_rf_b.mean():.3f} ± {scores_rf_b.std():.3f}")

# Permutation importance
X_train, X_test, y_train, y_test = train_test_split(
    X, y_severe, test_size=0.3, random_state=42,
)
clf_rf.fit(X_train, y_train)

perm = permutation_importance(
    clf_rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc",
)

imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df["feature"], imp_df["importance"], xerr=imp_df["std"],
        color="#e34a33", alpha=0.8)
ax.set_xlabel("Permutation Importance (AUC decrease)")
ax.set_title("Task B (severe_outcome) — Feature Importance")
plt.tight_layout()
plt.show()

print("\n=== Top 5 most important features (Task B) ===")
for _, row in imp_df.nlargest(5, "importance").iterrows():
    print(f"  {row['feature']:25s}  importance = {row['importance']:.4f}")

print("\n→ The important features for predicting severe outcomes may differ from those for predicting infection")
print("→ Exposure factors (shower_use) may matter for infection, but comorbidities matter more for severity")

## Question 3 (challenge): Three-model comparison + ROC curves

In [ ]:
# Three models
models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=500, random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocess", preprocess),
        ("model", GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ]),
}

# 70/30 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_infected, test_size=0.3, random_state=42,
)

# Train + AUC + ROC
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#2c7fb8", "#e34a33", "#41b6c4"]

print("=== Task A Test AUC ===")
for (name, clf), color in zip(models.items(), colors):
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=color, linewidth=2)
    print(f"  {name:25s}  AUC = {auc:.3f}")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Task A (infected)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print("\n→ On 280 rows, the three models usually perform very similarly")
print("→ A single 70/30 split gives unstable results; cross-validation is more reliable")
print("→ Interpret conclusions from a small sample conservatively, and avoid overstating model performance")

### Interpretation

- **class_weight**: On imbalanced data, `balanced` can improve recall of the minority class, but has limited impact on AUC
- **Task A vs Task B**: The key features for predicting infection (e.g. `shower_use`) and the key features for predicting severe outcomes (e.g. comorbidities) may differ, reflecting different causal mechanisms
- **Model choice**: 280 rows aren't enough to reveal the advantages of complex models. A simple model + proper cross-validation > a complex model + inappropriate evaluation
- **ML vs regression**: ML emphasizes prediction, regression emphasizes explanation. The two are complementary—when the feature importance ranking agrees with the direction of the adjusted OR, the conclusion is more convincing